[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Peewee, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/peewee-deep-dive.html)

# JSON Columns


## What you will be able to do

Put a whole document in one column with `JSONField`, and read it back as the dict it went in as.
Reach into it from a query with a path, and read the `->` and `$."a"."b"` that a path becomes. Say
why a comparison on a path is not a numeric comparison, and fix it with `as_int`, `as_float` or
`as_text`. Change one key of a document without rewriting the rest, with `set`, `update` and
`remove`. Recognize the same column declared as a `TextField`, which comes back a string and turns
every path query into a chain of comparisons that has nothing to do with the document.


## The idea

### The problem

`Edition.detail["printing"] > 9` looks like a number against a number. It is not. The path gives the
database a piece of JSON, and comparing a piece of JSON with the number 9 compares text, so the row
whose printing is 11 does not come back and the row whose printing is 3 might. No error, no warning,
an empty list.

What makes it hard to see is that the Python side is completely fine. `edition.detail["printing"]`
is an `int`, because peewee parsed the document on the way out. Only the comparison written into the
query behaves differently, and it is the one place you cannot check by printing the value.

### What a JSON column is

One column holding a whole document: nested dicts, lists, numbers, strings and booleans. peewee's
`JSONField` serializes on the way in and parses on the way out, so the attribute is an ordinary dict.
The database also understands the document, which is what makes a path query possible: you can filter
and sort on something inside the document without fetching every row into Python.

### Why it works that way

SQLite stores JSON as text and offers two extraction operators. `->` returns a piece of JSON, so
`data -> '$.printing'` on the value `11` gives the JSON `11`, which is still JSON and compares as
such. `->>` returns a SQL value. peewee's `[]` builds the first one, because a path can lead to a
dict or a list as easily as to a number, and there is no type to assume. `as_int` and its neighbors
build the second one and add a `CAST`, which is you supplying the type that the path could not.

### Where this shows up

Any column that holds settings, metadata, an API response or anything whose shape varies by row.
The trap appears the first time somebody filters on a number inside one, which is usually soon and
usually in a report that quietly comes back empty.

### What this notebook covers

Writing and reading a document. Reaching into it with a path, and the SQL that becomes. The
comparison that is not numeric, and the three casts that fix it. Changing one key without rewriting
the document. Asking how long a list inside it is. The same column declared `TextField` by mistake,
which fails in Python and, worse, does not fail in a query. Then the four failures, two silent.

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
from peewee import CharField, JSONField, Model, SqliteDatabase

db = SqliteDatabase(":memory:")


class Reading(Model):
    station = CharField()
    data = JSONField()

    class Meta:
        database = db


db.create_tables([Reading])
Reading.create(station="north", data={"n": 2})
Reading.create(station="south", data={"n": 10})

print("in Python, data['n'] is:", [(r.station, r.data["n"]) for r in Reading.select()])
print("where data['n'] > 9         ->",
      [r.station for r in Reading.select().where(Reading.data["n"] > 9)])
print("where data['n'].as_int() > 9 ->",
      [r.station for r in Reading.select().where(Reading.data["n"].as_int() > 9)])
```

```
in Python, data['n'] is: [('north', 2), ('south', 10)]
where data['n'] > 9         -> []
where data['n'].as_int() > 9 -> ['south']
```

The first line proves the values are numbers. The second line asks the database for the ones over
nine and gets none. The third asks the same question with the type supplied, and gets the answer the
first line said was there.


## Setup

Six imports, peewee installed and pinned, one model, and four editions.

- `peewee` is the library, and `Model`, `JSONField`, the other field classes and `SqliteDatabase`,
  from it, are what a model is written with
- `TextField` is here for one worked example, which declares a JSON column with it by mistake
- `OperationalError` is caught once, on an update that cannot be written the way it reads
- `json` writes a document by hand for that same example, which is what a `JSONField` does for you
- `subprocess`, `sys`, `version` and `PackageNotFoundError` install peewee 4.5.1 where the version is
  not that, as on Colab, whose 4.4.0 words some of these messages differently
- `sql` prints the SQL a query will send, with the values that go beside it

`EDITIONS` is four books with the things that vary by edition kept in a document rather than in
columns: a format, a page count, a printing number, a nested `rights` dict and a list of tags. That
is the shape a JSON column is for, where a `rights` table and a `tags` table would be more machinery
than the question deserves.


In [1]:
import json
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version

try:
    if version("peewee") != "4.5.1":                                # Colab has 4.4.0, whose wording differs
        raise PackageNotFoundError
except PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "peewee==4.5.1"], check=True)

import peewee
from peewee import (CharField, IntegerField, JSONField, Model, OperationalError, SqliteDatabase,
                    TextField)

def sql(query):
    """The SQL a query will send, and the values that go with it, on one line."""
    statement, values = query.sql()
    return " ".join(statement.split()) + (f"  {values}" if values else "")

EDITIONS = [                                                        # title, the edition as a document
    ("The Salt Road", {"format": "hardback", "pages": 312, "printing": 3,
                       "rights": {"regions": ["uk", "us"], "audio": True},
                       "tags": ["debut", "prize"]}),
    ("Nightjar", {"format": "paperback", "pages": 244, "printing": 11,
                  "rights": {"regions": ["uk"], "audio": False},
                  "tags": ["reprint"]}),
    ("Stone and Tide", {"format": "hardback", "pages": 501, "printing": 2,
                        "rights": {"regions": ["uk", "us", "ca"], "audio": True},
                        "tags": ["prize", "translated"]}),
    ("A Careful Fire", {"format": "paperback", "pages": 420, "printing": 9,
                        "rights": {"regions": ["ie", "uk"], "audio": False},
                        "tags": []}),
]

db = SqliteDatabase(":memory:", pragmas={"foreign_keys": 1})        # SQLite enforces nothing without this


class Edition(Model):
    """One row per edition, with everything that varies by edition in one JSON column."""

    title = CharField(max_length=80)
    detail = JSONField()

    class Meta:
        database = db


db.create_tables([Edition])
for title, detail in EDITIONS:
    Edition.create(title=title, detail=detail)

print("peewee", peewee.__version__, "|", Edition.select().count(), "editions |",
      "detail comes back as a", type(Edition.get().detail).__name__)


peewee 4.5.1 | 4 editions | detail comes back as a dict


## Worked examples

### A document in a column

What goes in comes back, with its types:


In [2]:
edition = Edition.get(Edition.title == "The Salt Road")

print("type:", type(edition.detail).__name__)
print("a string:", edition.detail["format"], "| a number:", edition.detail["pages"])
print("a nested dict:", edition.detail["rights"])
print("a list:", edition.detail["tags"], "| a bool:", edition.detail["rights"]["audio"])


type: dict
a string: hardback | a number: 312
a nested dict: {'regions': ['uk', 'us'], 'audio': True}
a list: ['debut', 'prize'] | a bool: True


The column itself is text as far as the table is concerned, and `JSONField` is the thing that
serializes on the way in and parses on the way out:


In [3]:
print(Edition._schema._create_table().query()[0])
raw = db.execute_sql("SELECT detail FROM edition WHERE title = ?", ("Nightjar",)).fetchone()[0]
print("as the column holds it:", raw[:60], "...")
print("as the model gives it: ", type(Edition.get(Edition.title == "Nightjar").detail).__name__)


CREATE TABLE IF NOT EXISTS "edition" ("id" INTEGER NOT NULL PRIMARY KEY, "title" VARCHAR(80) NOT NULL, "detail" TEXT NOT NULL)
as the column holds it: {"format":"paperback","pages":244,"printing":11,"rights":{"r ...
as the model gives it:  dict


### Reaching into it from a query

Indexing the field builds a path, and the path goes into the SQL:


In [4]:
hardbacks = Edition.select().where(Edition.detail["format"] == "hardback")
print(sql(hardbacks))
print([row.title for row in hardbacks])


SELECT "t1"."id", "t1"."title", "t1"."detail" FROM "edition" AS "t1" WHERE (("t1"."detail" -> ?) = json(?))  ['$."format"', '"hardback"']
['The Salt Road', 'Stone and Tide']


`->` is the extraction operator, and `$."format"` is the path. Nesting and list indexes work the same
way, and the path grows to match:


In [5]:
audio = Edition.select().where(Edition.detail["rights"]["audio"] == True)
first_region = Edition.select(Edition.title, Edition.detail["rights"]["regions"][0].as_text().alias("first"))

print(sql(audio)[sql(audio).index("WHERE"):])
print("audio rights:", [row.title for row in audio])
print()
print(sql(first_region)[:104])
print([(row.title, row.first) for row in first_region])


WHERE (("t1"."detail" -> ?) = json(?))  ['$."rights"."audio"', 'true']
audio rights: ['The Salt Road', 'Stone and Tide']

SELECT "t1"."title", ("t1"."detail" ->> ?) AS "first" FROM "edition" AS "t1"  ['$."rights"."regions"[0]'
[('The Salt Road', 'uk'), ('Nightjar', 'uk'), ('Stone and Tide', 'uk'), ('A Careful Fire', 'ie')]


### The comparison that is not numeric

Everything above compared against a string or a boolean, and worked. A number does not:


In [6]:
plain = Edition.select().where(Edition.detail["printing"] > 9)
typed = Edition.select().where(Edition.detail["printing"].as_int() > 9)

print("printings, in Python:", [(row.title, row.detail["printing"]) for row in Edition.select()])
print()
print("printing > 9          ->", [row.title for row in plain])
print("printing.as_int() > 9 ->", [row.title for row in typed])


printings, in Python: [('The Salt Road', 3), ('Nightjar', 11), ('Stone and Tide', 2), ('A Careful Fire', 9)]

printing > 9          -> []
printing.as_int() > 9 -> ['Nightjar']


Two rows have a printing over nine and the first query found neither. The SQL says why:


In [7]:
print("plain:", sql(plain).split("WHERE ")[1])
print("typed:", sql(typed).split("WHERE ")[1])


plain: (("t1"."detail" -> ?) > json(?))  ['$."printing"', '9']
typed: (CAST(("t1"."detail" ->> ?) AS INTEGER) > ?)  ['$."printing"', 9]


`("detail" -> ?) > json(?)` is JSON against JSON, which SQLite compares as text, and as text `"11"`
sorts before `"9"`. `CAST(("detail" ->> ?) AS INTEGER) > ?` is a number against a number.

There are three casts, and they are the only way to tell the database what a path holds:


In [8]:
print("as_int  :", [row.title for row in
                    Edition.select().where(Edition.detail["pages"].as_int() > 400)])
print("as_float:", [row.title for row in
                    Edition.select().where(Edition.detail["printing"].as_float() < 2.5)])
print("as_text :", [row.title for row in
                    Edition.select().where(Edition.detail["format"].as_text() == "paperback")])


as_int  : ['Stone and Tide', 'A Careful Fire']
as_float: ['Stone and Tide']
as_text : ['Nightjar', 'A Careful Fire']


`as_text` is worth noticing: the string comparison worked without it further up, because a JSON
string compared with a JSON string is the same comparison either way. It is the numeric ones that
need saying. Ordering has the same rule, and the printing numbers show it because they are not all
the same width:


In [9]:
by_text = [row.detail["printing"] for row in
           Edition.select().order_by(Edition.detail["printing"])]
by_number = [row.detail["printing"] for row in
             Edition.select().order_by(Edition.detail["printing"].as_int())]

print("printings ordered by the path:", by_text)
print("printings ordered by as_int():", by_number)


printings ordered by the path: [11, 2, 3, 9]
printings ordered by as_int(): [2, 3, 9, 11]


### Changing one key

Writing the whole dict back is the obvious way, and it rewrites the document:


In [10]:
edition = Edition.get(Edition.title == "Nightjar")
edition.detail = {**edition.detail, "printing": 12}
edition.save()
print("after a whole write:", Edition.get(Edition.title == "Nightjar").detail["printing"])


after a whole write: 12


That reads the document into Python, changes it and sends all of it back, which is a lost update
waiting to happen if anything else is writing the same row. The database can do it in place instead.
`set` puts a value at a path, `update` merges a dict into the document, and `remove` takes a path
out:


In [11]:
for label, change in (
        ("set", Edition.detail["printing"].set(13)),
        ("update", Edition.detail.update({"format": "paperback", "reissued": 2024})),
        ("remove", Edition.detail["tags"].remove())):
    query = Edition.update({Edition.detail: change}).where(Edition.title == "Nightjar")
    print(f"  {label:<7} {sql(query).split('SET ')[1][:62]}")
    query.execute()

print()
print("Nightjar now:", Edition.get(Edition.title == "Nightjar").detail)


  set     "detail" = json_set("edition"."detail", ?, ?) WHERE ("edition"
  update  "detail" = json_patch("edition"."detail", json(?)) WHERE ("edi
  remove  "detail" = json_remove("edition"."detail", ?) WHERE ("edition"

Nightjar now: {'format': 'paperback', 'pages': 244, 'printing': 13, 'rights': {'regions': ['uk'], 'audio': False}, 'reissued': 2024}


`json_set`, `json_patch` and `json_remove` are SQLite's own functions, and the row never traveled to
Python. The `reissued` key is new, which is what `update` does to a key that was not there.

### How long is the list

`length` asks the database, which is how you filter on it:


In [12]:
counted = Edition.select(Edition.title, Edition.detail["rights"]["regions"].length().alias("regions"))
print(sql(counted)[:108])
for row in counted.order_by(Edition.title):
    print(f"  {row.title:<16} {row.regions} region{'' if row.regions == 1 else 's'}")

print()
print("more than one region:",
      [row.title for row in Edition.select().where(
          Edition.detail["rights"]["regions"].length() > 1)])


SELECT "t1"."title", json_array_length("t1"."detail", ?) AS "regions" FROM "edition" AS "t1"  ['$."rights"."
  A Careful Fire   2 regions
  Nightjar         1 region
  Stone and Tide   3 regions
  The Salt Road    2 regions

more than one region: ['The Salt Road', 'Stone and Tide', 'A Careful Fire']


### The same column, declared TextField

A `JSONField` is a `TextField` that knows what is in it. Declare the column as a plain `TextField`
and everything still stores, and nothing works:


In [13]:
class Plain(Model):
    title = CharField(max_length=80)
    detail = TextField()                                            # a JSON column by mistake

    class Meta:
        database = db


db.create_tables([Plain])
for title, detail in EDITIONS:
    Plain.create(title=title, detail=json.dumps(detail))            # serialized by hand

row = Plain.get(Plain.title == "The Salt Road")
print("type:", type(row.detail).__name__)
print("the value:", row.detail[:46], "...")


type: str
the value: {"format": "hardback", "pages": 312, "printing ...


A string, because nothing parsed it. In Python that fails loudly the moment it is used as a document,
which is the good case and the first of the Common errors below. In a query it does not fail at all:


In [14]:
good = Edition.select().where(Edition.detail["format"] == "hardback")
bad = Plain.select().where(Plain.detail["format"] == "hardback")

print("JSONField ->", [row.title for row in good])
print("TextField ->", [row.title for row in bad])
print()
print("JSONField sql:", sql(good).split("WHERE ")[1])
print("TextField sql:", sql(bad).split("WHERE ")[1])


JSONField -> ['The Salt Road', 'Stone and Tide']
TextField -> []

JSONField sql: (("t1"."detail" -> ?) = json(?))  ['$."format"', '"hardback"']
TextField sql: (("t1"."detail" = ?) = ?)  ['format', 'hardback']


Nothing found, and a `WHERE` clause with no JSON in it. Indexing a plain field is not a path at all:
peewee read `Plain.detail["format"]` as the comparison `detail = 'format'`, and then `== "hardback"`
compared the true or false that came out of that with a string.

So the answer depends on how a chain of comparisons happens to collapse, and never on the document.
Nothing found is not even the worst case: with different literals the same chain collapses the other
way and returns the whole table, which the Common errors below show side by side. This is the shape
of the `&` in the **Selecting Rows** notebook again, an operator meaning something other than it
reads, with the query running either way.

### On PostgreSQL

`JSONField` works on both backends. PostgreSQL also has a binary JSON type that indexes well, and
peewee exposes it as `BinaryJSONField` in `playhouse.postgres_ext`, with the same `[]` paths and the
same casts. Which SQL each backend produces is the **SQLite and PostgreSQL** notebook's subject.

### When to reach for which

| What you want | How to write it |
|---|---|
| a document in a column | `JSONField()` |
| a value inside it, in Python | `row.detail["rights"]["audio"]` |
| a filter on a string or a boolean | `Model.detail["format"] == "hardback"` |
| a filter on a number | `Model.detail["pages"].as_int() > 400` |
| an order by a number | `.order_by(Model.detail["pages"].as_int())` |
| one key changed, in the database | `Model.detail["printing"].set(13)` |
| several keys merged in | `Model.detail.update({...})` |
| a key taken out | `Model.detail["tags"].remove()` |
| the length of a list inside it | `Model.detail["tags"].length()` |
| the same, on PostgreSQL | `BinaryJSONField` from `playhouse.postgres_ext` |

The default for reading is the plain path, and the default for anything numeric is a cast. The
default for writing is `set` or `update` rather than reading the document into Python and sending it
back, because the second one loses whatever somebody else wrote in between.

### A report, finished

Everything above, as the thing it was for: a report that filters and sorts on values inside the
document, and a change written in place.


In [15]:
def wide_release(least_regions=2, least_pages=300):
    """Editions sold in several regions, longest first, with their audio rights."""
    query = (Edition.select()
                    .where((Edition.detail["rights"]["regions"].length() >= least_regions)
                           & (Edition.detail["pages"].as_int() >= least_pages))
                    .order_by(Edition.detail["pages"].as_int().desc()))
    return [(row.title, row.detail["pages"], row.detail["rights"]["regions"],
             row.detail["rights"]["audio"]) for row in query]


for title, pages, regions, audio in wide_release():
    print(f"  {title:<16} {pages:>3} pages  {regions}  audio: {audio}")

Edition.update({Edition.detail: Edition.detail["rights"]["audio"].set(True)}).where(
    Edition.title == "A Careful Fire").execute()
print()
print("after granting audio rights:", wide_release(least_regions=2, least_pages=400))


  Stone and Tide   501 pages  ['uk', 'us', 'ca']  audio: True
  A Careful Fire   420 pages  ['ie', 'uk']  audio: False
  The Salt Road    312 pages  ['uk', 'us']  audio: True

after granting audio rights: [('Stone and Tide', 501, ['uk', 'us', 'ca'], True), ('A Careful Fire', 420, ['ie', 'uk'], True)]


Both conditions are inside the document, both have their own parentheses, and the numeric one has its
cast. The change at the end reached two levels into the document and rewrote nothing else.

### Where each part came from

| In the report | What it relies on | The section that showed it |
|---|---|---|
| `detail["rights"]["regions"].length()` | a list's length, counted by the database | How long is the list |
| `detail["pages"].as_int() >= least_pages` | a number compared as a number | The comparison that is not numeric |
| `&` with parentheses around each condition | the operator rules | **Selecting Rows** |
| `.order_by(...as_int().desc())` | an order that is numeric too | The comparison that is not numeric |
| `row.detail["rights"]["audio"]` | the document parsed on the way out | A document in a column |
| `detail["rights"]["audio"].set(True)` | one key changed in place | Changing one key |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/peewee-deep-dive/08-json-columns-solutions.ipynb).

**1.** Print each edition's title and format by reading the document in Python, and then find the
paperbacks with a query.


In [16]:
# your code here


**2.** Find the editions of more than 400 pages, first without a cast and then with one, and print
both answers and both `WHERE` clauses.


In [17]:
# your code here


**3.** Order the editions by printing number, correctly, and print the numbers beside the titles.


In [18]:
# your code here


**4.** Add a `price` key to every edition with one statement that does not read the documents into
Python, then print the documents.


In [19]:
# your code here


**5.** Print each edition with the number of tags it has, and find the ones with none.


In [20]:
# your code here


**6.** Write the same document into a `TextField` column and show two things: what Python does when
you index the value, and what a path query returns.


In [21]:
# your code here


## Common errors

### No error, and nothing found for a number that is plainly there: a path compared as JSON


In [22]:
big = Edition.select().where(Edition.detail["printing"] > 9)

print("printings in the table:", sorted(row.detail["printing"] for row in Edition.select()))
print("rows found:", [row.title for row in big])
print("the clause:", sql(big).split("WHERE ")[1])


printings in the table: [2, 3, 9, 13]
rows found: []
the clause: (("t1"."detail" -> ?) > json(?))  ['$."printing"', '9']


A printing of 13 is in the table and the query found nothing, because `->` returns JSON and JSON
compared with a number is compared as text, where `"13"` sorts before `"9"`.

Worse than empty is possible, and it is why this cannot be caught by glancing at a result. The same
uncast comparison against a threshold whose text happens to sort correctly gives the right answer,
so a filter checked once on data like that is wrong on data it meets later:


In [23]:
lucky = Edition.select().where(Edition.detail["pages"] > 400)       # no cast, right answer
print("pages > 400, no cast:", [(row.title, row.detail["pages"]) for row in lucky])

right = Edition.select().where(Edition.detail["printing"].as_int() > 9)
print("printing > 9, cast  :", [(row.title, row.detail["printing"]) for row in right])

print("text order  :", [row.detail["printing"] for row in
                        Edition.select().order_by(Edition.detail["printing"])])
print("number order:", [row.detail["printing"] for row in
                        Edition.select().order_by(Edition.detail["printing"].as_int())])


pages > 400, no cast: [('Stone and Tide', 501), ('A Careful Fire', 420)]
printing > 9, cast  : [('Nightjar', 13)]
text order  : [13, 2, 3, 9]
number order: [2, 3, 9, 13]


### TypeError: string indices must be integers, not 'str'


In [24]:
row = Plain.get(Plain.title == "The Salt Road")
row.detail["format"]


TypeError: string indices must be integers, not 'str'

The column was declared `TextField`, so the value came back as the string it is stored as and was
never parsed. This is the friendly half of that mistake: it raises at the first use, and the fix is
one word in the model.

If the model cannot be changed, `json.loads` is the manual version of what `JSONField` does:


In [25]:
parsed = json.loads(row.detail)
print("after json.loads:", type(parsed).__name__, "|", parsed["format"])
print("what the model should say:", "detail = JSONField()")


after json.loads: dict | hardback
what the model should say: detail = JSONField()


### No error, and an answer that has nothing to do with the document: a path on a plain column


In [26]:
for label, condition in (
        ("detail['rights']['audio'] == True", Plain.detail["rights"]["audio"] == True),
        ("detail['rights']['regions'][0] == 1", Plain.detail["rights"]["regions"][0] == 1)):
    found = Plain.select().where(condition)
    print(f"  {label:<36} {len(found)} of {Plain.select().count()} rows")
    print(f"  {'':<36} {sql(found).split('WHERE ')[1]}")


  detail['rights']['audio'] == True    0 of 4 rows
                                       ((("t1"."detail" = ?) = ?) = ?)  ['rights', 'audio', True]
  detail['rights']['regions'][0] == 1  4 of 4 rows
                                       (((("t1"."detail" = ?) = ?) = ?) = ?)  ['rights', 'regions', 0, 1]


Not one of those rows was checked for audio rights, and the second spelling returned the whole
table. On a plain field, `[]` is not a JSON path: peewee turns each index into an equality test, so
the clause is a chain of comparisons collapsing to a true or a false that is then compared with your
value. Whether that comes out as every row or no rows is decided by the literals in the chain.

This is the dangerous half of the `TextField` mistake. The Python side raises and gets fixed. The
query side returns a plausible number of rows and looks like a filter that worked:


In [27]:
print("JSONField, the same filter:",
      [row.title for row in Edition.select().where(Edition.detail["rights"]["audio"] == True)])


JSONField, the same filter: ['The Salt Road', 'Stone and Tide', 'A Careful Fire']


### peewee.OperationalError: near "->": syntax error


In [28]:
Edition.update({Edition.detail["printing"]: 5}).where(Edition.title == "Nightjar").execute()


OperationalError: near "->": syntax error

A path can be read in a `WHERE` clause and cannot be assigned to in a `SET` clause, because SQL has
nowhere to put it: `SET "detail" -> '$.printing' = 5` is not a statement. The column is what gets
assigned, and the new value is the whole document with that one key changed, which is what `set`
builds:


In [29]:
query = Edition.update({Edition.detail: Edition.detail["printing"].set(5)}).where(
    Edition.title == "Nightjar")
print(sql(query).split("SET ")[1][:70])
query.execute()
print("printing now:", Edition.get(Edition.title == "Nightjar").detail["printing"])


"detail" = json_set("edition"."detail", ?, ?) WHERE ("edition"."title"
printing now: 5


## Recap

- `JSONField` stores a document in one column, serializing on the way in and parsing on the way out,
  so the attribute is an ordinary dict.
- Indexing the field in a query builds a path, which becomes `->` and a `$."key"` string in the SQL.
  Nesting and list indexes extend the path.
- A path returns JSON, so comparing it with a number compares text. `as_int`, `as_float` and
  `as_text` add the `CAST` that makes it a value, and they are needed in `order_by` as well as in
  `where`.
- String and boolean comparisons happen to work without a cast, which is why the numeric case is
  surprising when it arrives.
- `set`, `update` and `remove` change a document in the database, without reading it into Python and
  writing all of it back.
- `length` counts a list inside the document, and can be filtered on.
- The same column declared `TextField` comes back a string. Indexing it in Python raises. Indexing
  it in a query builds a chain of equality tests that returns every row.
- `BinaryJSONField` in `playhouse.postgres_ext` is the PostgreSQL version.


## What is next

The **FTS5Model and SearchField** notebook is about searching text rather than matching it: a
virtual table built for it, `SearchField`, ranking by relevance, and `web_query`, which turns a
search box into a small query language whether or not anybody meant it to be one.


---

&#8592; **Previous:** [prefetch and Load](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/peewee-deep-dive/07-prefetch-and-load.ipynb)  &nbsp;·&nbsp;  [Peewee, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/peewee-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [FTS5Model and SearchField](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/peewee-deep-dive/09-fts5model-and-searchfield.ipynb) &#8594;
